# 02 — Deep Agent with Microsoft Foundry Web Search

## Goal

Extend our baseline Deep Agent with real public-web research.

We will use:

- Deep Agents as the agent harness
- Microsoft Foundry for model inference
- Microsoft Foundry Web Search for current web grounding
- Entra ID for authentication

We intentionally still do NOT add:

- tracing
- evaluation
- persistence
- deployment
- Azure AI Search
- memory

---

## New architecture

```
User
  ↓
Deep Agent
  ↓
LangGraph
  ↓
AzureAIOpenAIApiChatModel
  ↓
Foundry Responses API
  ↓
Web Search
  ↓
Public Web
```

In [1]:
import os

from dotenv import load_dotenv

load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
MODEL_DEPLOYMENT = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]

In [2]:
from azure.identity import DefaultAzureCredential
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel

credential = DefaultAzureCredential()

model = AzureAIOpenAIApiChatModel(
    project_endpoint=PROJECT_ENDPOINT,
    credential=credential,
    model=MODEL_DEPLOYMENT,
)

print(type(model))

<class 'langchain_azure_ai.chat_models.openai.AzureAIOpenAIApiChatModel'>


## Create and Test the Foundry Web Search Tool

In [3]:
from langchain_azure_ai.tools.builtin import WebSearchTool

web_search = WebSearchTool()

print(type(web_search))

<class 'langchain_azure_ai.tools.builtin._tools.WebSearchTool'>


C:\Users\shchitt\AppData\Local\Temp\ipykernel_35252\2030170112.py:3: ExperimentalWarning: WebSearchTool is currently in preview and is subject to change. This preview is provided without a service-level agreement, and we don't recommend it for production workloads. Certain features might not be supported or might have constrained capabilities. For more information, see https://azure.microsoft.com/support/legal/preview-supplemental-terms
  web_search = WebSearchTool()


In [4]:
model_with_web_search = model.bind_tools([web_search])

print(type(model_with_web_search))

<class 'langchain_core.language_models.chat_models._ChatModelBinding'>


In [7]:
response = model_with_web_search.invoke(
    """
    What are the most recent major updates to Microsoft Foundry?
    Give me only 3 important updates and cite your sources.
    """
)

response.content

[{'id': 'ws_0fce0e96d05bcf97006a9d04cffed08195b6595806a1b41c36',
  'action': {'type': 'search',
   'queries': ['Microsoft Foundry major updates 2026',
    'Microsoft Foundry announcements 2026 Microsoft Build Foundry',
    'Microsoft Foundry product update release notes',
    '"Microsoft Foundry" "update"'],
   'query': 'Microsoft Foundry major updates 2026'},
  'status': 'completed',
  'type': 'web_search_call',
  'response_id': 'resp_0fce0e96d05bcf97006a9d04cf2bfc8195837554785ca6bae1'},
 {'id': 'ws_0fce0e96d05bcf97006a9d04d280ac81958e0ea5a803dfb5d6',
  'action': {'type': 'open_page',
   'url': 'https://learn.microsoft.com/en-us/azure/foundry/whats-new-foundry'},
  'status': 'completed',
  'type': 'web_search_call',
  'response_id': 'resp_0fce0e96d05bcf97006a9d04cf2bfc8195837554785ca6bae1'},
 {'type': 'text',
  'text': '1) **Hosted agents for production runtimes (Foundry Agent Service) matured at Build 2026** — Microsoft highlighted hosted agents with sandboxed sessions, state + files

In [8]:
for i, block in enumerate(response.content):
    print(f"\n--- BLOCK {i} ---")
    print(block)


--- BLOCK 0 ---
{'id': 'ws_0fce0e96d05bcf97006a9d04cffed08195b6595806a1b41c36', 'action': {'type': 'search', 'queries': ['Microsoft Foundry major updates 2026', 'Microsoft Foundry announcements 2026 Microsoft Build Foundry', 'Microsoft Foundry product update release notes', '"Microsoft Foundry" "update"'], 'query': 'Microsoft Foundry major updates 2026'}, 'status': 'completed', 'type': 'web_search_call', 'response_id': 'resp_0fce0e96d05bcf97006a9d04cf2bfc8195837554785ca6bae1'}

--- BLOCK 1 ---
{'id': 'ws_0fce0e96d05bcf97006a9d04d280ac81958e0ea5a803dfb5d6', 'action': {'type': 'open_page', 'url': 'https://learn.microsoft.com/en-us/azure/foundry/whats-new-foundry'}, 'status': 'completed', 'type': 'web_search_call', 'response_id': 'resp_0fce0e96d05bcf97006a9d04cf2bfc8195837554785ca6bae1'}

--- BLOCK 2 ---
{'type': 'text', 'text': '1) **Hosted agents for production runtimes (Foundry Agent Service) matured at Build 2026** — Microsoft highlighted hosted agents with sandboxed sessions, state 

In [9]:
for i, block in enumerate(response.content_blocks):
    print(f"\n--- CONTENT BLOCK {i} ---")
    print(block)


--- CONTENT BLOCK 0 ---
{'type': 'server_tool_call', 'name': 'web_search', 'args': {'type': 'search', 'queries': ['Microsoft Foundry major updates 2026', 'Microsoft Foundry announcements 2026 Microsoft Build Foundry', 'Microsoft Foundry product update release notes', '"Microsoft Foundry" "update"'], 'query': 'Microsoft Foundry major updates 2026'}, 'id': 'ws_0fce0e96d05bcf97006a9d04cffed08195b6595806a1b41c36', 'response_id': 'resp_0fce0e96d05bcf97006a9d04cf2bfc8195837554785ca6bae1'}

--- CONTENT BLOCK 1 ---
{'type': 'server_tool_result', 'tool_call_id': 'ws_0fce0e96d05bcf97006a9d04cffed08195b6595806a1b41c36', 'status': 'success'}

--- CONTENT BLOCK 2 ---
{'type': 'server_tool_call', 'name': 'web_search', 'args': {'type': 'open_page', 'url': 'https://learn.microsoft.com/en-us/azure/foundry/whats-new-foundry'}, 'id': 'ws_0fce0e96d05bcf97006a9d04d280ac81958e0ea5a803dfb5d6', 'response_id': 'resp_0fce0e96d05bcf97006a9d04cf2bfc8195837554785ca6bae1'}

--- CONTENT BLOCK 3 ---
{'type': 'server

### Verify the search actually happened

In [10]:
for block in response.content_blocks:
    block_type = block.get("type")
    print(block_type)

server_tool_call
server_tool_result
server_tool_call
server_tool_result
text


In [11]:
for block in response.content_blocks:
    if block.get("type") == "text":
        print("TEXT:")
        print(block.get("text"))

        print("\nANNOTATIONS:")
        for annotation in block.get("annotations", []):
            print(annotation)

TEXT:
1) **Hosted agents for production runtimes (Foundry Agent Service) matured at Build 2026** — Microsoft highlighted hosted agents with sandboxed sessions, state + filesystem access, and broader framework flexibility (with GA expected in early July 2026). ([devblogs.microsoft.com](https://devblogs.microsoft.com/foundry/whats-new-in-microsoft-foundry-build-2026/))

2) **Foundry IQ expanded into a broader “knowledge plane”** — updates included serverless retrieval, new knowledge sources/knowledge bases, Web IQ, and agentic retrieval improvements to reduce custom RAG plumbing. ([devblogs.microsoft.com](https://devblogs.microsoft.com/foundry/whats-new-in-microsoft-foundry-build-2026/))

3) **Model Router got major new observability + broader availability (Sept 2026)** — per-request routing metadata (preview) to expose serving model and routing attempts/fallbacks, plus expansion to **32 Azure regions** and model refreshes (e.g., GPT‑5.6 family and Claude Opus 4.8). ([learn.microsoft.com

## Add Web Search to Deep Agents

Microsoft's guidance for LangChain agents using server-side tools is to bind the server-side tool to the model and expose the tool to the agent configuration.

In [13]:
research_instructions = """
You are an expert research assistant.

Your job is to research questions thoroughly and produce concise,
evidence-grounded answers.

## Research behavior

- Use web search whenever current or externally verifiable information is needed.
- Prefer authoritative and primary sources when possible.
- Search more than once when the first search does not fully answer the question.
- Distinguish established facts from your interpretation.
- Include source citations in your final answer when web research was used.

You also have the standard Deep Agents capabilities for planning,
filesystem-based context management, and subagent delegation.
"""

In [15]:
from langchain_core.language_models import BaseChatModel

print("Raw model type:")
print(type(model))

print("\nBound model type:")
print(type(model_with_web_search))

print("\nRaw model is BaseChatModel:")
print(isinstance(model, BaseChatModel))

print("\nBound model is BaseChatModel:")
print(isinstance(model_with_web_search, BaseChatModel))

Raw model type:
<class 'langchain_azure_ai.chat_models.openai.AzureAIOpenAIApiChatModel'>

Bound model type:
<class 'langchain_core.language_models.chat_models._ChatModelBinding'>

Raw model is BaseChatModel:
True

Bound model is BaseChatModel:
False


In [16]:
from deepagents import create_deep_agent

tools = [web_search]

research_agent = create_deep_agent(
    model=model,   # IMPORTANT: raw BaseChatModel, not model_with_web_search
    tools=tools,
    system_prompt=research_instructions,
)

print(type(research_agent))

<class 'langgraph.graph.state.CompiledStateGraph'>


Architecture:

```
                    create_deep_agent
                           │
               ┌───────────┴────────────┐
               │                        │
          Deep Agent                Foundry model
          middleware                    │
               │                   WebSearchTool
      ┌────────┼────────┐               │
      │        │        │               ▼
   planning filesystem task        Foundry Responses
                                      │
                                      ▼
                                  public web
```

In [18]:
research_question = """
Research the current state of Microsoft Foundry Hosted Agents.

Explain:

1. What problem Hosted Agents solve.
2. What kinds of custom agent frameworks they can host.
3. How they differ conceptually from server-managed prompt agents.
4. Two important limitations or operational considerations.

Use current Microsoft documentation and give me a concise technical summary
with citations.
"""

result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": research_question,
            }
        ]
    }
)

In [21]:
for i, message in enumerate(result["messages"]):
    print(f"\n{'=' * 80}")
    print(f"MESSAGE {i}: {type(message).__name__}")
    print(f"{'=' * 80}")

    if hasattr(message, "content_blocks"):
        for block in message.content_blocks:
            print(block)
    else:
        print(message)


MESSAGE 0: HumanMessage
{'type': 'text', 'text': '\nResearch the current state of Microsoft Foundry Hosted Agents.\n\nExplain:\n\n1. What problem Hosted Agents solve.\n2. What kinds of custom agent frameworks they can host.\n3. How they differ conceptually from server-managed prompt agents.\n4. Two important limitations or operational considerations.\n\nUse current Microsoft documentation and give me a concise technical summary\nwith citations.\n'}

MESSAGE 1: AIMessage
{'type': 'server_tool_call', 'name': 'web_search', 'args': {'type': 'search', 'queries': ['Microsoft Foundry Hosted Agents documentation', '"Hosted Agents" "Foundry" Microsoft', 'Microsoft Foundry hosted agents frameworks supported', 'Azure AI Foundry Hosted Agents limitations'], 'query': 'Microsoft Foundry Hosted Agents documentation'}, 'id': 'ws_08edacc13f16873f006a9d0b357a848193afc84cadd0e4b9eb', 'response_id': 'resp_08edacc13f16873f006a9d0b3484e88193a75d733e6b418364'}
{'type': 'server_tool_result', 'tool_call_id': 

In [23]:
def inspect_agent_activity(messages):
    for i, message in enumerate(messages):
        print(f"\n### Message {i}: {type(message).__name__}")

        # Client-side Deep Agent tool calls
        tool_calls = getattr(message, "tool_calls", None)
        if tool_calls:
            print("LOCAL / AGENT TOOL CALLS:")
            for call in tool_calls:
                print(" ", call)

        # Server-side content blocks
        content_blocks = getattr(message, "content_blocks", None)
        if content_blocks:
            print("CONTENT BLOCKS:")
            for block in content_blocks:
                block_type = block.get("type")
                print(f"  - {block_type}")
                
                
inspect_agent_activity(result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
CONTENT BLOCKS:
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - text


In [33]:
from IPython.display import display, Markdown

display(Markdown(result['messages'][-1].content[-1]['text']))

### 1) What problem Hosted Agents solve

**Microsoft Foundry Hosted Agents** (in Foundry Agent Service) are meant for teams building agents with *code-first/open-source frameworks* who don’t want to also build and operate the surrounding production platform. Microsoft positions Hosted Agents as a managed runtime that takes on cross‑cutting operational work such as **containerization/web server setup, security, state persistence, scaling, observability, and lifecycle/rollbacks**, so you can focus on agent logic and orchestration. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))

Conceptually: you ship a container with your agent code; Foundry provisions per-session compute, routes requests to your container, and provides an agent identity and managed state/ops around it. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))

---

### 2) What kinds of custom agent frameworks they can host

Hosted Agents are **bring-your-own-code**: you can run “any framework … or custom code” inside the hosted sandbox/container, and Microsoft documentation explicitly calls out **Microsoft Agent Framework, LangGraph, Semantic Kernel, or custom code**. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))

Operationally, the hosted service exposes **OpenAI-compatible “Responses protocol” endpoints**, and also supports the **Invocations protocol** (useful when you want to accept webhooks or non‑OpenAI payloads). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))

---

### 3) How they differ conceptually from server-managed prompt agents

A useful mental model from Microsoft’s docs is:

- **Hosted Agents = “bring your own runtime + stateful sandbox.”** You deploy your **own containerized agent code**, and Foundry provides the managed hosting plane (identity, scaling, session sandboxing + persisted filesystem, observability). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))
- **Prompt-based / server-managed prompt agents = “definition-only agents.”** You primarily configure prompts/tools/behavior declaratively and rely on the service’s built-in orchestration rather than shipping and operating a custom web service. Microsoft’s Hosted Agents guidance explicitly frames Hosted Agents as the choice when you need **“Bring your own code … rather than prompt-only definitions”** and when you need more control over compute, protocols, and stateful workloads. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))

---

### 4) Two important limitations / operational considerations

1) **Sessions are stateful, persistent, and time-bounded (and scale-to-zero has implications).**  
   Hosted Agents use a **session** as the unit of state and isolation; the platform persists the session filesystem (**`$HOME` and uploaded files**) across turns and idle periods, but **sessions persist “for up to 30 days”** and compute is deprovisioned after an **idle timeout (5–60 minutes, default 15)**. Design implications: you must treat session state as durable only within those constraints, and ensure reconnection/resumption behavior is correct. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/manage-hosted-sessions))

2) **Regional and feature availability constraints (tools vary by region/model) + service quotas.**  
   Foundry Agent Service is regional, and Microsoft notes **tool availability differs by region** (example given: **file search not available** in some regions). Separately, the service enforces **quotas/limits** on artifacts like file uploads/messages/tools; you need to design to those boundaries and handle limit errors. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/limits-quotas-regions))

If you want, I can turn this into an architecture decision checklist (Hosted Agents vs in-process vs prompt-only) tailored to your workload (protocol needs, state model, networking, compliance).

### Test whether Deep Agents actually plans

In [34]:
complex_question = """
Research three viable architectural approaches for building
a production enterprise research agent on Microsoft Foundry.

Compare them on:

- amount of custom code
- control over orchestration
- deployment model
- observability
- suitability for complex long-running research

Use current Microsoft documentation.
Create a structured comparison and recommend when each approach makes sense.
"""

In [35]:
complex_result = research_agent.invoke(
    {"messages": [{"role": "user", "content": complex_question}]}
)

inspect_agent_activity(complex_result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
CONTENT BLOCKS:
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - text


In [36]:
display(Markdown(complex_result['messages'][-1].content[-1]['text']))

## Three viable production architectures on Microsoft Foundry (with current Microsoft docs)

I’m interpreting “Microsoft Foundry” as **Microsoft (Azure AI) Foundry** and “enterprise research agent” as an agent that does **tool-using research/RAG, iterative planning, citations, and multi-step execution**, often exceeding a single request/response.

### The three approaches
1. **Foundry Agent Service – Prompt/Hosted Agent (managed orchestration)**
2. **Microsoft Agent Framework (MAF) code-first agent hosted on Foundry Agent Service (“Hosted agents”)**
3. **MAF Durable Workflows / Durable Agents (long-running) hosted on Azure Functions or “bring your own compute”, optionally surfaced in Foundry**

These map cleanly to three common enterprise needs: *fastest path*, *maximum code control with managed hosting*, and *durable long-running research*.

---

## Structured comparison

| Dimension | 1) Foundry Agent Service “prompt/managed agent” (built-in orchestration) | 2) MAF code-first agent hosted as a **Foundry Hosted Agent** | 3) MAF **Durable** Agents / Durable Workflows (Azure Functions or BYOC) |
|---|---|---|---|
| **Amount of custom code** | **Low–Medium**. You configure the agent in Foundry; you add tools via **function calling** or queue-based **Azure Functions tools**. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)) | **Medium–High**. You implement the agent in code (MAF), package, deploy; Foundry hosts it. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent)) | **High**. You implement workflows/orchestrations and durability wiring (Durable extension). ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/azure-functions)) |
| **Control over orchestration** | **Moderate**. You get agent/tool loop behavior and configuration, but less control over advanced routing, multi-agent supervision, custom retry policies, and bespoke state machines. (Best for “single agent + tools” patterns.) ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)) | **High**. You own orchestration logic in code (multi-agent patterns, planners, routing, custom memory/state), while Foundry handles hosting concerns. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent)) | **Very High**. Designed for durable orchestration: checkpointing, recovery, distributed scaling, and explicit workflow graphs/pipelines. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/azure-functions)) |
| **Deployment model** | **Managed in Foundry**. Agent runs in Foundry Agent Service; tools can be REST/function-calling endpoints or async queue-triggered Azure Functions. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling)) | **Containerized hosted deployment on Foundry managed infrastructure** (Foundry Hosted Agents). Foundry handles scaling/session persistence/security/lifecycle. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent)) | **Azure Functions** (serverless) or **bring-your-own compute** using Durable Task extension; you operate the runtime more directly than (1)/(2). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/durable-task/sdks/durable-agents-microsoft-agent-framework)) |
| **Observability** | **Strong, Foundry-native**: agent tracing to **Azure Monitor Application Insights** via OpenTelemetry; captures latency, exceptions, prompts, retrieval, etc. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup)) | **Strong**: MAF has built-in OpenTelemetry (GenAI semantic conventions); plus Foundry-hosted agents are designed to cover observability and production ops. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/agents/observability)) | **Strongest for long-running**: same MAF OpenTelemetry + durable runtime visibility; better fit for tracking progress across hours/days with checkpoints/retries. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/agents/observability)) |
| **Suitability for complex, long-running research** | **Fair–Good** for “multi-step within one session” and async tool calls (e.g., queue-based tools). Less ideal for multi-hour workflows requiring durable checkpoints and robust recovery. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/azure-functions)) | **Good** when you need complex logic but can keep runs bounded; you can implement your own persistence and resumability, but you’re responsible for the design. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents)) | **Excellent**: durability is a first-class feature (persist sessions, checkpoint progress, recover from failures, scale across hosts). Best fit for “research jobs” that must survive restarts/timeouts. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/azure-functions)) |

---

## What each architecture looks like (high level)

### 1) Foundry Agent Service (managed agent + tools)
- Build agent in Foundry; extend capabilities via:
  - **Function calling** (sync tool invocation patterns). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/function-calling))  
  - **Azure Functions queue-based tools** (async tool execution, scale-to-zero). ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/how-to/tools/azure-functions))  
- Add **Foundry tracing** to Application Insights for production debugging/monitoring. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup))  

**When this shines:** fastest path to production for a research assistant that primarily does *search/retrieval + tool calls + response composition*.

---

### 2) Microsoft Agent Framework (code-first) hosted on Foundry (“Hosted Agent”)
- Write the agent in MAF (C#/Python, etc.), implement your orchestration logic and toolchain.
- Deploy as a **Foundry Hosted Agent**, where Foundry provides managed hosting/scaling/session persistence/security/lifecycle. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/foundry-hosted-agent))  
- Use MAF’s OpenTelemetry observability and/or Foundry’s operational features. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/agents/observability))  

**When this shines:** you need more sophisticated orchestration than the managed agent model (custom routers, supervisor patterns, multiple specialist sub-agents, bespoke memory), but still want Foundry to handle production hosting.

---

### 3) MAF Durable Agents / Durable Workflows (long-running research jobs)
- Use the **Durable Extension** to make agent sessions and orchestrations durable (checkpointing, recovery, distributed scaling). ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/azure-functions))  
- Host on **Azure Functions** (serverless) or your own compute. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/durable-task/sdks/durable-agents-microsoft-agent-framework))  
- Model multi-step research as a workflow/pipeline (MAF workflows). ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/workflows/agents-in-workflows))  

**When this shines:** truly long-running enterprise research (e.g., “monitor 20 sources, run extraction, dedupe, synthesize, produce report, get human review, retry failed steps”) where resumability and reliability matter more than minimal code.

---

## Recommendations (when each makes sense)

### Choose **Approach 1 (Foundry managed agent + tools)** when:
- You want **minimal code** and fastest delivery.
- Your “research” fits a conversational pattern with tool calls, even if some tools run async (queue-based Azure Functions).
- You want **Foundry-native tracing** quickly in Application Insights. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/trace-agent-setup))  

### Choose **Approach 2 (MAF hosted agent on Foundry)** when:
- You need **high orchestration control** (custom multi-agent strategies, routing, planner/supervisor patterns).
- You still want **managed hosting** and platform help with scaling/session concerns. ([learn.microsoft.com](https://learn.microsoft.com/en-us/azure/foundry/agents/concepts/hosted-agents))  
- You want strong, standards-based observability via **OpenTelemetry GenAI conventions**. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/agents/observability))  

### Choose **Approach 3 (MAF Durable workflows/agents)** when:
- Research tasks are **long-running, multi-stage, failure-prone**, or must survive restarts/timeouts.
- You need **durability guarantees** (checkpointing, replay/recovery, distributed execution).
- You’re willing to accept more engineering to gain **reliability and explicit workflow control**. ([learn.microsoft.com](https://learn.microsoft.com/en-us/agent-framework/hosting/azure-functions))  

---

## One practical decision rule
- If it’s **interactive Q&A with tools** → start with **(1)**.
- If it’s **complex orchestration but still “session-ish”** → **(2)**.
- If it’s a **research job** (minutes→hours, needs checkpoints, retries, and resumability) → **(3)**.

If you tell me your expected research duration (seconds vs minutes vs hours), number of tools/systems, and whether you need human-in-the-loop steps, I can map that to a concrete reference architecture (components, data stores, and recommended telemetry setup).

# Two kinds of tools now exist

Our system contains two fundamentally different tool execution paths.

## 1. Deep Agent / local tools

Examples:

- write_todos
- write_file
- read_file
- task

Execution:

```
Model
  ↓
LangGraph / Deep Agents
  ↓
Python-side tool execution
  ↓
ToolMessage
  ↓
Model
```

These tools belong to the agent harness.


## 2. Foundry server-side tools

Example:

- WebSearchTool

Execution:

```
Deep Agent
  ↓
Foundry model request
  ↓
Foundry Responses API
  ↓
web_search executes server-side
  ↓
results are returned inside the model response
```

These tools belong to the model/platform runtime.


## Mental model

```
                 DEEP AGENT
                     │
          ┌──────────┴──────────┐
          │                     │
    Harness tools          Foundry model
          │                     │
    LangGraph/Python        Server tools
          │                     │
   write_file etc.          web_search
```

### Does the agent use the filesystem during research?

In [37]:
filesystem_question = """
Research the current Microsoft guidance for building and operating
custom agents in Foundry.

Do the following:

1. Research the topic from current Microsoft documentation.
2. Organize your important findings in /research_notes.md.
3. Read the notes back.
4. Produce a concise final architecture summary with citations.

Do not simply answer from memory.
"""

In [38]:
filesystem_result = research_agent.invoke(
    {"messages": [{"role": "user", "content": filesystem_question}]}
)

In [40]:

inspect_agent_activity(filesystem_result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
LOCAL / AGENT TOOL CALLS:
  {'name': 'write_file', 'args': {'file_path': '/research_notes.md', 'content': '# Microsoft guidance for building and operating custom agents in Microsoft Foundry (Agent Service)\n\nDate researched: 2026-09-06\n\n## Scope & definitions\n- **Foundry Agent Service** is Microsoft’s managed platform for **building, deploying, and scaling AI agents**. It supports multiple build styles:\n  - **Prompt agents** (declarative; no container management)\n  - **Hosted agents** (bring-your-own containerized code/framework)\n  - **External agents** that you run elsewhere but invoke Foundry via APIs (e.g., Responses API)  \ue200cite\ue202turn1view0\ue201\n- Core platform components called out in Microsoft docs:\n  - **Agent runtime** (hosts/scales prompt + hosted agents; manages conversations, tool calls, lifecycle) \ue200cite\ue202turn1view0\ue201\n  - **Toolboxes** (curated tool sets exposed vi

In [42]:
display(Markdown(filesystem_result['messages'][-1].content[-1]['text']))

## `/research_notes.md` (read back)

# Microsoft guidance for building and operating custom agents in Microsoft Foundry (Agent Service)

Date researched: 2026-09-06

## Scope & definitions
- **Foundry Agent Service** is Microsoft’s managed platform for **building, deploying, and scaling AI agents**. It supports multiple build styles:
  - **Prompt agents** (declarative; no container management)
  - **Hosted agents** (bring-your-own containerized code/framework)
  - **External agents** that you run elsewhere but invoke Foundry via APIs (e.g., Responses API) 
- Core platform components called out in Microsoft docs:
  - **Agent runtime** (hosts/scales prompt + hosted agents; manages conversations, tool calls, lifecycle) 
  - **Toolboxes** (curated tool sets exposed via a managed MCP endpoint; centralized auth/governance/versioning; tools include web search, file search, code interpreter, MCP servers, custom functions) 
  - **Models** via the Foundry model catalog (swap models without changing agent code) 
  - **Observability** (end-to-end tracing/metrics/evaluations; Application Insights integration) 
  - **Optimization** via agent optimizer (preview) to improve instructions/skills/tool descriptions/model selection based on evaluations/production signals 
  - **Identity & security**: Microsoft Entra identity + RBAC + content filters + VNet isolation 
  - **Publishing**: versioning + stable endpoints + distribution (Teams / Microsoft 365 Copilot / Entra Agent Registry) 

## When Microsoft says to use “Hosted agents” (custom code)
- Hosted agents address cross-cutting production concerns you otherwise build yourself (containerization, web server setup, security, memory persistence, scaling, instrumentation, rollbacks), especially across heterogeneous environments. 
- Choose hosted agents over prompt agents when you need: 
  - **Bring your own code/framework** (Agent Framework, LangGraph, Semantic Kernel, custom)
  - **Custom protocols** (webhooks / non-OpenAI payloads via the Invocations protocol)
  - **Control compute** (CPU/memory) for the agent sandbox
  - **Stateful workloads** (persist files/state across turns via `$HOME` and `/files`)
  - **Resilient long-lived work** (preserve in-progress work across interruptions; replay streamed results)

## Hosted agent deployment & runtime model (what you build vs what Foundry runs)
- Deployment flow (hosted agents): package agent into a **container image**, push to **Azure Container Registry**, then deploy into Agent Service. 
- At deploy time, Agent Service:
  - pulls the image,
  - assigns a dedicated **Microsoft Entra ID (agent identity)**,
  - exposes a dedicated **endpoint** for the agent. 
- At runtime, Agent Service provisions compute for a session and routes requests to your container. Your code can call:
  - Foundry models,
  - Toolbox tools,
  - downstream Azure services using the agent identity.
  - Platform handles **scaling, session state persistence, observability, lifecycle mgmt**. 

## Operating hosted agents (lifecycle, versioning, endpoints)
- After deployment you can: view status, create new **versions**, select the version served by an endpoint, monitor logs, delete agents. Platform manages the container lifecycle. 
- Compute scaling behavior:
  - compute is provisioned on request and deprovisioned after an **idle timeout** (5–60 minutes; default 15). 
  - scaling is separate from endpoint state: you can **disable/enable an endpoint** to take an agent offline/online; you do not manually start/stop compute. 
- Operational access patterns in docs include REST calls (`az rest`) to Foundry Agent Service endpoints and SDKs (Python `azure-ai-projects`, JS `@azure/ai-projects`), authenticated with Entra (note the `--resource` audience requirement in examples). 

## Networking & private connectivity guidance (BYO VNet)
- For bring-your-own VNet in Agent Service, customer is responsible for:
  - delegated subnet sizing,
  - IP allocation planning,
  - understanding traffic flows and capacity signals. 
- Architecture includes two zones:
  - Microsoft-managed Foundry platform network (endpoint, micro-VM host layer for hosted agents, tools service, data proxy)
  - customer VNet (delegated subnet where micro-VMs + data proxy consume IPs; private endpoint subnet to storage/DB/Key Vault). 
- Request flows:
  - Hosted agent: client → Foundry endpoint → **Micro VM** (/invoke) → tools service → data proxy → customer resources via private endpoints
  - Prompt agent: client → Foundry endpoint → tools service → data proxy → customer resources via private endpoints (no micro-VM) 

## Observability & evaluation guidance
- Microsoft emphasizes **evaluation** as essential to quality/safety before deployment and for ongoing improvement; encourages acceptance thresholds (example: 85% task adherence). 
- Recommended eval approach for agents:
  - generate a **rubric evaluator** from the agent’s context as primary measure
  - layer built-in evaluators (content safety and other risks)
  - create datasets and run evals; interpret results and integrate into workflow. 
- Observability in Foundry includes tracing/metrics and Application Insights integration (called out at platform level). 

## Enterprise governance / cross-org operations
### Foundry Control Plane (operate across many projects)
- Foundry Control Plane provides **centralized management and observability** for agents across supported platforms and infrastructures. 
- Control Plane offers an **agent inventory** across projects within a subscription; automatically discovers supported agents; shows metadata + health indicators and can surface metrics like error rate, token usage, runs, estimated cost if observability configured. 

### Registering custom agents (agents running outside Foundry runtime)
- You can register custom agents (running in Azure compute services or other clouds) so Foundry Control Plane can manage/observe them. 
- Registration model:
  - Foundry uses **Azure API Management** as a proxy to control access and monitor activity.
  - Requirements include: exclusive/reachable agent endpoint; supported protocols **HTTP** or **A2A**; optionally emit telemetry using **OpenTelemetry semantic conventions for generative AI**. 
  - After registration, Control Plane generates a new URL for users to communicate with the agent. 

## Publishing agents for production consumption (and current state of “Agent Applications”)
- Microsoft docs describe **Agent Applications** as a (legacy) publishing experience; Microsoft recommends migrating to the new agent endpoint/publishing model (separate doc). 
- Publishing rationale: project-level development isn’t meant for broad distribution because project access shares context/permissions; publishing creates a production-ready Azure resource with a stable endpoint and its own auth/RBAC boundary. 
- When you publish an agent version (legacy model): Foundry creates an **Agent Application resource** with its own invocation URL, authentication policy, **unique Entra agent identity** and **Entra agent blueprint**; a **Deployment** child resource references a specific agent version and supports start/stop lifecycle. 

## Microsoft Agent 365 integration (org-wide governance)
- Microsoft Agent 365 is described as an enterprise control plane treating agents as first-class **Microsoft Entra identities** via **Entra Agent ID**, enabling authz/governance workflows on agent identities. 
- Agent 365 provides pillars: registry, access control (RBAC/ABAC + Conditional Access), visualization, interoperability (M365), security (Defender + Purview). 
- Foundry integration includes **registry sync** so published Foundry agents appear in Agent 365 registry automatically. 


---

## Concise architecture summary (custom/hosted agents in Foundry)

**Recommended production shape (per Microsoft docs):**

1) **Build (your responsibility)**  
- Implement agent logic in your preferred framework; containerize it for **Hosted agents**.   
- Define tool access via **Toolboxes** (managed MCP endpoint) rather than embedding ad-hoc credentials in code.   

2) **Deploy (Foundry + you)**  
- Push image to **Azure Container Registry** and deploy to **Foundry Agent Service**.   
- Foundry assigns an **Entra-based agent identity** and exposes an **agent endpoint**; your code uses that identity to call models/tools/Azure resources.   

3) **Runtime execution (Foundry responsibility, with your integrations)**  
- Requests hit the Foundry endpoint and are routed to your hosted-agent micro-VM/container; Foundry manages conversations/tool-call orchestration, scaling, and state persistence.   
- For private enterprise access, use **BYO VNet** and private endpoints; plan delegated subnet capacity and understand hosted-agent vs prompt-agent traffic flow differences.   

4) **Operate (Foundry surfaces controls; you own SLOs and improvements)**  
- Manage versions/endpoints, monitor health/logs, and rely on platform scaling with an **idle timeout** policy (5–60 minutes; default 15).   
- Implement continuous quality/safety loops with **evaluations** (rubric + safety evaluators) and feed results into iteration/optimization.   

5) **Govern at scale (enterprise)**  
- Use **Foundry Control Plane** for centralized inventory/health/metrics across subscriptions/projects; register **non-Foundry-hosted** agents via APIM proxy for consistent management and observability.   
- For organization-wide governance/distribution, integrate with **Microsoft Agent 365 / Entra Agent ID** so agents are treated as managed identities with policy controls. 

### If we don't explicitly tell it to search

In [43]:
implicit_search_result = research_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
What are the major capabilities of Microsoft Foundry that were
introduced or substantially changed recently?

Summarize only the most consequential changes for an enterprise AI architect.
""",
            }
        ]
    }
)

In [44]:
inspect_agent_activity(implicit_search_result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
CONTENT BLOCKS:
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - text


### Planning question

In [45]:
# planning_question = """
# Conduct a structured investigation of how an enterprise could
# productionize a custom LangGraph/Deep Agents research agent on
# Microsoft Foundry.

# You need to independently investigate:

# 1. Hosted Agent deployment/runtime
# 2. Authentication and managed identity
# 3. Toolboxes and MCP
# 4. Networking/private connectivity
# 5. Observability
# 6. Evaluation
# 7. Versioning and production operations

# Develop an architecture recommendation that connects all seven areas,
# identifies dependencies between them, and highlights current limitations.

# Use current Microsoft documentation.
# """

In [49]:
planning_question = """
Conduct a structured investigation of how an enterprise could
productionize a custom LangGraph/Deep Agents research agent on
Microsoft Foundry.

You need to independently investigate:

1. Hosted Agent deployment/runtime
2. Authentication and managed identity
3. Toolboxes and MCP
4. Networking/private connectivity
5. Observability
6. Evaluation
7. Versioning and production operations

Create a plan, develop an architecture recommendation that connects all seven areas,
identifies dependencies between them, and highlights current limitations.

Use current Microsoft documentation.
"""

In [50]:
planning_result = research_agent.invoke(
    {"messages": [{"role": "user", "content": planning_question}]}
)

inspect_agent_activity(planning_result["messages"])


### Message 0: HumanMessage
CONTENT BLOCKS:
  - text

### Message 1: AIMessage
CONTENT BLOCKS:
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - server_tool_call
  - server_tool_result
  - text


# What we learned

## 1. Foundry Web Search replaces Tavily cleanly

Original Deep Agents Quickstart:

```
Deep Agent
  ↓
Python internet_search()
  ↓
Tavily
```

Our architecture:

```
Deep Agent
  ↓
Foundry-backed model
  ↓
Foundry Web Search
  ↓
public web
```


## 2. Web Search is a server-side tool

This differs from ordinary Python tools.

Local tool:

```
Model → LangGraph → Python function → ToolMessage → Model
```

Foundry Web Search:

```
Model → Foundry Responses API → Web Search → Model response
```


## 3. Deep Agents still owns the harness

Foundry gives us:

- model inference
- web search

Deep Agents gives us:

- planning
- filesystem/context management
- subagent capability
- long-running agent harness

LangGraph executes the outer agent workflow.


## 4. A Deep Agent is not automatically better

For simple research:

```
model + web search
```

may be sufficient.

Deep Agents becomes valuable when tasks require:

- decomposition
- many research steps
- intermediate state
- context offloading
- delegation
- long-running work


## 5. Current architecture

```
                     User
                       ↓
                  Deep Agent
                       ↓
               LangGraph runtime
                  ↙         ↘
       harness tools       Foundry model
      planning/files            ↓
                           Web Search
                               ↓
                           Public Web
```


## Next milestone

Notebook 03 will make this system observable.

We will add:

```
Deep Agent
   ↓
OpenTelemetry
   ↓
Application Insights
   ↓
Foundry Tracing
```

Then instead of manually inspecting messages,
we will inspect the complete agent trajectory as telemetry.